## 1. Install required libraries

In [ ]:
!pip -q install xgboost gensim prefixspan tabulate

## 2. Import libraries

In [ ]:
import re
import warnings
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from prefixspan import PrefixSpan

from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from xgboost import XGBClassifier
from tabulate import tabulate

warnings.filterwarnings("ignore")

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

print("Libraries imported successfully!")

## 3. Upload and load the dataset



In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file was uploaded.")

file_name = next(iter(uploaded))

df = pd.read_csv(file_name, sep=None, engine="python")

print("Dataset loaded successfully!")
print("\nFirst five records:")
display(df.head())

print("\nDataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

## 4. Select text and label columns




In [ ]:
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"

if TEXT_COLUMN not in df.columns or LABEL_COLUMN not in df.columns:
    raise ValueError(
        f"Expected columns '{TEXT_COLUMN}' and '{LABEL_COLUMN}'. "
        f"Your CSV columns are: {df.columns.tolist()}. "
        "Update TEXT_COLUMN and LABEL_COLUMN to match your file."
    )

df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN]).copy()

X_raw = df[TEXT_COLUMN].astype(str)
y_original = df[LABEL_COLUMN].astype(str).str.strip()

print("Number of emails:", len(df))
print("\nClass distribution:")
print(y_original.value_counts())

## 5. Text preprocessing

In [ ]:
def preprocess_text(text):
    text = str(text)
    text = re.sub(r"[^a-zA-Z\\s]", " ", text)
    text = text.lower()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)


X = X_raw.apply(preprocess_text)

print("Text preprocessing completed!")
print("\nSample preprocessed emails:")
display(X.head())

## 6. Encode labels and split the dataset




In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_original)

if len(label_encoder.classes_) != 2:
    raise ValueError(
        "This notebook currently expects a binary classification dataset "
        "with exactly two labels."
    )

print("Label mapping:")
for encoded_value, original_label in enumerate(label_encoder.classes_):
    print(f"{original_label} = {encoded_value}")

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train_text))
print("Testing samples:", len(X_test_text))

## 7. TF-IDF feature extraction

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("TF-IDF feature extraction completed!")
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 8. Word2Vec feature extraction



In [ ]:
tokenized_train = [text.split() for text in X_train_text]
tokenized_test = [text.split() for text in X_test_text]

word2vec_model = Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    epochs=10,
    seed=42
)

def get_average_word2vec(text):
    words = text.split()
    word_vectors = [
        word2vec_model.wv[word]
        for word in words
        if word in word2vec_model.wv
    ]

    if not word_vectors:
        return np.zeros(100)

    return np.mean(word_vectors, axis=0)


X_train_word2vec = np.array([
    get_average_word2vec(text) for text in X_train_text
])

X_test_word2vec = np.array([
    get_average_word2vec(text) for text in X_test_text
])

print("Word2Vec feature extraction completed!")
print("Training shape:", X_train_word2vec.shape)
print("Testing shape:", X_test_word2vec.shape)

In [ ]:
train_sequences = [text.split() for text in X_train_text]

min_support = max(2, int(0.01 * len(train_sequences)))

ps = PrefixSpan(train_sequences)
ps.maxlen = 3

frequent_patterns = ps.frequent(min_support)

frequent_patterns = sorted(
    frequent_patterns,
    key=lambda item: item[0],
    reverse=True
)[:100]

patterns = [pattern for support, pattern in frequent_patterns]

print("Minimum support:", min_support)
print("Number of selected frequent patterns:", len(patterns))
print("\nTop frequent patterns:")

for support, pattern in frequent_patterns[:10]:
    print("Pattern:", " ".join(pattern), "| Support:", support)

In [ ]:
def contains_pattern(sequence, pattern):
    # Check whether the pattern occurs as a subsequence in the email.
    pattern_index = 0

    for word in sequence:
        if word == pattern[pattern_index]:
            pattern_index += 1

            if pattern_index == len(pattern):
                return 1

    return 0


def create_pattern_features(texts, selected_patterns):
    sequences = [text.split() for text in texts]

    if not selected_patterns:
        return csr_matrix((len(sequences), 0), dtype=np.float32)

    features = [
        [
            contains_pattern(sequence, pattern)
            for pattern in selected_patterns
        ]
        for sequence in sequences
    ]

    return csr_matrix(np.asarray(features, dtype=np.float32))


X_train_prefix = create_pattern_features(X_train_text, patterns)
X_test_prefix = create_pattern_features(X_test_text, patterns)

print("PrefixSpan feature extraction completed!")
print("Training shape:", X_train_prefix.shape)
print("Testing shape:", X_test_prefix.shape)

## 10. Combine feature representations



In [ ]:
X_train_combined = hstack([
    X_train_tfidf,
    csr_matrix(X_train_word2vec),
    X_train_prefix
]).tocsr()

X_test_combined = hstack([
    X_test_tfidf,
    csr_matrix(X_test_word2vec),
    X_test_prefix
]).tocsr()

print("Combined feature extraction completed!")
print("Training features:", X_train_combined.shape)
print("Testing features:", X_test_combined.shape)

## 11. Feature scaling


In [ ]:
scaler = StandardScaler(with_mean=False)

X_train = scaler.fit_transform(X_train_combined)
X_test = scaler.transform(X_test_combined)

print("Feature scaling completed!")

## 12. Train Logistic Regression

In [ ]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression training completed!")

## 13. Train XGBoost

In [ ]:
xgb = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost training completed!")

## 14. Identify the spam label



In [ ]:
SPAM_LABEL = None  # Example: "spam" or "1". Set this if automatic detection is incorrect.

spam_candidates = {"spam", "spam email", "junk", "1", "true", "yes", "phishing"}

if SPAM_LABEL is not None:
    matches = [
        i for i, label in enumerate(label_encoder.classes_)
        if str(label).strip().lower() == str(SPAM_LABEL).strip().lower()
    ]
    if not matches:
        raise ValueError(f"SPAM_LABEL '{SPAM_LABEL}' was not found in the dataset labels.")
    spam_index = matches[0]
else:
    matches = [
        i for i, label in enumerate(label_encoder.classes_)
        if str(label).strip().lower() in spam_candidates
    ]
    if matches:
        spam_index = matches[0]
    else:
        print("Could not identify the spam label automatically.")
        print("For now, class 1 is treated as spam. Change SPAM_LABEL above if needed.")
        spam_index = 1

print("Spam class:", label_encoder.classes_[spam_index])

## 15. Compare model performance

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    return [
        model_name,
        round(accuracy_score(y_true, y_pred), 4),
        round(precision_score(
            y_true, y_pred, pos_label=spam_index, zero_division=0
        ), 4),
        round(recall_score(
            y_true, y_pred, pos_label=spam_index, zero_division=0
        ), 4),
        round(f1_score(
            y_true, y_pred, pos_label=spam_index, zero_division=0
        ), 4)
    ]


results = [
    evaluate_model(y_test, y_pred_lr, "Logistic Regression"),
    evaluate_model(y_test, y_pred_xgb, "XGBoost")
]

headers = ["Model", "Accuracy", "Precision", "Recall", "F1-Score"]

print("\nSpam Email Classification Results:\n")
print(tabulate(results, headers=headers, tablefmt="grid"))

## 16. Classification reports

In [ ]:
target_names = [str(label) for label in label_encoder.classes_]

print("LOGISTIC REGRESSION REPORT")
print(classification_report(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    target_names=target_names,
    zero_division=0
))

print("\nXGBOOST REPORT")
print(classification_report(
    y_test,
    y_pred_xgb,
    labels=[0, 1],
    target_names=target_names,
    zero_division=0
))

## 17. Accuracy comparison

In [ ]:
lr_accuracy = accuracy_score(y_test, y_pred_lr)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

plt.figure(figsize=(8, 5))
plt.bar(
    ["Logistic Regression", "XGBoost"],
    [lr_accuracy, xgb_accuracy],
    color=["green", "purple"]
)
plt.xlabel("Classification Models")
plt.ylabel("Accuracy")
plt.title("Spam Email Classification - Accuracy Comparison")
plt.ylim(0, 1)
plt.show()

## 18. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=[0, 1])
cm_xgb = confusion_matrix(y_test, y_pred_xgb, labels=[0, 1])

sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=axes[0]
)
axes[0].set_title("Logistic Regression Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Actual Label")

sns.heatmap(
    cm_xgb,
    annot=True,
    fmt="d",
    cmap="Purples",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=axes[1]
)
axes[1].set_title("XGBoost Confusion Matrix")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

## 19. ROC curve comparison


In [ ]:
y_score_lr = lr.predict_proba(X_test)[:, spam_index]
y_score_xgb = xgb.predict_proba(X_test)[:, spam_index]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_score_lr, pos_label=spam_index)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_score_xgb, pos_label=spam_index)

roc_auc_lr = auc(fpr_lr, tpr_lr)
roc_auc_xgb = auc(fpr_xgb, tpr_xgb)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr_lr, tpr_lr,
    label=f"Logistic Regression (AUC = {roc_auc_lr:.2f})",
    color="green"
)
plt.plot(
    fpr_xgb, tpr_xgb,
    label=f"XGBoost (AUC = {roc_auc_xgb:.2f})",
    color="purple"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Spam Email Classification")
plt.legend()
plt.grid()
plt.show()

## 20. Identify the best-performing model

In [ ]:
if lr_accuracy > xgb_accuracy:
    best_model = "Logistic Regression"
elif xgb_accuracy > lr_accuracy:
    best_model = "XGBoost"
else:
    best_model = "Both models have equal accuracy"

print("Logistic Regression Accuracy:", round(lr_accuracy, 4))
print("XGBoost Accuracy:", round(xgb_accuracy, 4))
print("\nBest model based on accuracy:", best_model)